In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Mengatur seed untuk reproduktifitas
np.random.seed(42)

# --- 1. Membuat Data Sintetis untuk Ilustrasi ---


n_samples = 500
# Fitur: biaya perjalanan (cost) dan waktu perjalanan (time_spent)
cost = np.random.uniform(5, 50, n_samples)
time_spent = np.random.uniform(15, 90, n_samples)

df_data = pd.DataFrame({'cost': cost, 'time_spent': time_spent})

# Utilitas untuk setiap pilihan (semakin tinggi, semakin disukai)
# Mobil sebagai kategori dasar
util_car = -0.05 * df_data['cost'] - 0.02 * df_data['time_spent'] + np.random.gumbel(loc=0.5, scale=1, size=n_samples)
# Bus Biru dan Bus Merah dibuat memiliki koefisien dan utilitas yang sangat mirip
util_blue_bus = -0.08 * df_data['cost'] - 0.03 * df_data['time_spent'] + np.random.gumbel(loc=0.3, scale=1, size=n_samples)
util_red_bus = -0.07 * df_data['cost'] - 0.03 * df_data['time_spent'] + np.random.gumbel(loc=0.35, scale=1, size=n_samples) # Sangat mirip dengan bus biru

# Menentukan pilihan berdasarkan utilitas tertinggi
choices = np.argmax(np.array([util_car, util_blue_bus, util_red_bus]), axis=0)
choice_map = {0: 'Car', 1: 'Blue Bus', 2: 'Red Bus'}
df_data['choice'] = pd.Categorical(pd.Series(choices).map(choice_map))


print("--- Distribusi Pilihan Awal ---")
print(df_data['choice'].value_counts())
print("\n" + "="*50 + "\n")


# --- 2. Melatih Model Penuh (Full Model) ---
print("--- Estimasi Model Penuh (Full Model) ---")
full_model = smf.mnlogit('choice ~ cost + time_spent', data=df_data).fit(disp=False) # disp=False untuk menekan output iterasi
print(full_model.summary())
print("\n" + "="*50 + "\n")


# --- 3. Melatih Model Subset (Restricted Model) ---

print("--- Estimasi Model Subset (tanpa 'Red Bus') ---")
df_subset = df_data[df_data['choice'] != 'Red Bus'].copy()
df_subset['choice'] = df_subset['choice'].cat.remove_unused_categories()

restricted_model = smf.mnlogit('choice ~ cost + time_spent', data=df_subset).fit(disp=False)
print(restricted_model.summary())
print("\n" + "="*50 + "\n")


# --- 4. Membandingkan Koefisien untuk Mengidentifikasi Pelanggaran IIA ---

# Ekstrak koefisien untuk 'Blue Bus' dari kedua model
coef_full_blue_bus = full_model.params.filter(like='Blue Bus', axis=0)
coef_restricted_blue_bus = restricted_model.params.filter(like='Blue Bus', axis=0)

comparison_df = pd.DataFrame({
    'Full Model Coeff': coef_full_blue_bus,
    'Restricted Model Coeff': coef_restricted_blue_bus
})
comparison_df['Difference'] = comparison_df['Full Model Coeff'] - comparison_df['Restricted Model Coeff']

print("--- Perbandingan Koefisien untuk Pilihan 'Blue Bus' ---")
print(comparison_df)
print("\n" + "="*50 + "\n")


--- Distribusi Pilihan Awal ---
choice
Car         323
Red Bus      93
Blue Bus     84
Name: count, dtype: int64


--- Estimasi Model Penuh (Full Model) ---


ValueError: endog has evaluated to an array with multiple columns that has shape (500, 3). This occurs when the variable converted to endog is non-numeric (e.g., bool or str).